
# QubitPath AI — Quantum Computing Education Platform

This Colab notebook builds a complete **Streamlit prototype** for a three-level quantum-computing education platform.

### Included in the prototype

- Entry, intermediate, and advanced learning pathways
- Recorded-learning library with lesson notes, activities, references, and official external resources
- Live-tutoring schedule and prototype booking workflow for Zoom, Microsoft Teams, or Google Meet
- Qiskit-powered quantum circuit laboratory and mini-games
- Short quizzes with explanations, experience points, and learner progress tracking
- Machine-learning progress forecasting
- Deep neural-network learner-support risk prediction
- Reinforcement-learning-style adaptive teaching recommendations using an epsilon-greedy contextual bandit
- “Professor Qubit” AI character using transparent retrieval from a curated quantum knowledge base
- Interactive learning analytics dashboard and downloadable learner report
- Responsible-AI, privacy, accessibility, and production-deployment guidance

> **Important:** The included models use synthetic demonstration data. They are educational prototypes and must not be used for consequential grading, admissions, or student discipline.



## 1. Install the required packages

Run this cell once in a fresh Google Colab runtime. The notebook pins compatible major versions for repeatability while remaining suitable for current Colab Python runtimes.


In [ ]:
%pip install -q \
  "streamlit==1.59.2" \
  "qiskit[visualization]~=2.5.0" \
  "scikit-learn>=1.5,<2" \
  "plotly>=5.24,<7" \
  "pandas>=2.2,<3" \
  "numpy>=1.26,<3" \
  "matplotlib>=3.8,<4" \
  "joblib>=1.4,<2"


## 2. Verify the runtime

In [ ]:

import sys
import platform
import numpy as np
import pandas as pd
import sklearn
import streamlit
import qiskit

print("Python:", sys.version)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Streamlit:", streamlit.__version__)
print("Qiskit:", qiskit.__version__)



## 3. Prototype architecture

**Learning layer** → leveled curriculum, recordings/resources, live tutoring, quizzes, games, quantum lab  
**Analytics layer** → progress metrics, engagement trends, knowledge-skill profile, forecast dashboard  
**AI layer** → random-forest progress model, deep neural-network support-risk model, retrieval tutor, adaptive bandit  
**Quantum layer** → Qiskit circuits, statevector simulation, sampled measurements, circuit visualization  
**Deployment layer** → Streamlit app generated by this notebook and launched through a temporary Colab tunnel

The app intentionally separates **model advice** from **human tutoring decisions**. Tutor review remains part of the live-session and learner-support workflow.



## 4. Generate the Streamlit application

The next cell writes the complete app to `app.py`. You can open the generated file from Colab's Files panel, or download it using the later export cell.


In [ ]:
APP_CODE = '# QubitPath AI - Streamlit application\n# Generated from the accompanying Colab prototype.\n# No API keys are required for the default demo.\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport random\nfrom datetime import date, datetime, timedelta\nfrom typing import Any\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport plotly.graph_objects as go\nimport streamlit as st\nfrom qiskit import QuantumCircuit\nfrom qiskit.quantum_info import Statevector\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\n\nAPP_TITLE = "QubitPath AI"\nAUTHOR_NAME = "Suman Poola"\nMENTOR_NAME = "Dr. Qingyang Xiao"\nLEVELS = ["Entry", "Intermediate", "Advanced"]\nLEVEL_TO_NUM = {"Entry": 1, "Intermediate": 2, "Advanced": 3}\nIBM_COURSES_URL = "https://quantum.cloud.ibm.com/learning/en/courses"\nQISKIT_DOCS_URL = "https://qiskit.qotlabs.org/docs/"\n\nst.set_page_config(\n    page_title=f"{APP_TITLE} | Quantum Learning",\n    page_icon="⚛️",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\nst.markdown(\n    """\n    <style>\n      :root { --qp-purple:#6f4bf2; --qp-cyan:#13c8c8; --qp-ink:#172033; }\n      .stApp { background: linear-gradient(180deg,#f8f7ff 0%,#f4fbff 100%); }\n      .block-container { padding-top: 1.2rem; padding-bottom: 3rem; max-width: 1400px; }\n      .hero {\n        padding: 1.45rem 1.6rem; border-radius: 24px; color: white;\n        background: radial-gradient(circle at 10% 20%,#8a6cff 0,#6f4bf2 35%,#27357f 100%);\n        box-shadow: 0 18px 50px rgba(61,54,140,.20); margin-bottom: 1rem;\n      }\n      .hero h1 { margin:0; font-size:2.35rem; }\n      .hero p { margin:.45rem 0 0; opacity:.94; font-size:1.04rem; }\n      .glass-card {\n        background:rgba(255,255,255,.92); border:1px solid rgba(111,75,242,.12);\n        padding:1rem 1.1rem; border-radius:18px; box-shadow:0 8px 26px rgba(30,38,90,.07);\n        min-height:145px;\n      }\n      .pill { display:inline-block; padding:.22rem .62rem; border-radius:999px;\n        background:#eee9ff; color:#4b35b4; font-weight:700; font-size:.78rem; margin-right:.3rem; }\n      .success-box { background:#ecfff7; border-left:5px solid #20a977; padding:.8rem 1rem; border-radius:12px; }\n      .info-box { background:#eef8ff; border-left:5px solid #368bea; padding:.8rem 1rem; border-radius:12px; }\n      .warning-box { background:#fff8e7; border-left:5px solid #e4a11b; padding:.8rem 1rem; border-radius:12px; }\n      .small-muted { color:#667085; font-size:.86rem; }\n      div[data-testid="stMetric"] { background:white; border:1px solid #e8e8f5; padding:.7rem; border-radius:16px; }\n      section[data-testid="stSidebar"] { background:linear-gradient(180deg,#171b36,#202958); color:white; }\n      section[data-testid="stSidebar"] label, section[data-testid="stSidebar"] p { color:#f4f5ff !important; }\n      section[data-testid="stSidebar"] h1, section[data-testid="stSidebar"] h2, section[data-testid="stSidebar"] h3 { color:white; }\n      .project-credits-sidebar {\n        margin:.45rem 0 1rem; padding:.72rem .78rem; border-radius:14px;\n        background:rgba(255,255,255,.09); border:1px solid rgba(255,255,255,.16);\n        line-height:1.42; font-size:.86rem;\n      }\n      .project-credits-sidebar .credit-label { color:#c9cdef; font-size:.72rem; font-weight:700; text-transform:uppercase; letter-spacing:.05em; }\n      .project-credits-sidebar .mentor-credit { margin-top:.52rem; padding-top:.52rem; border-top:1px solid rgba(255,255,255,.13); }\n      .project-credits-page { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:.8rem; margin:.2rem 0 1.25rem; }\n      .credit-card { background:rgba(255,255,255,.94); border:1px solid rgba(111,75,242,.16); padding:1rem 1.1rem; border-radius:16px; box-shadow:0 7px 22px rgba(30,38,90,.06); }\n      .credit-role { color:#6043d8; font-size:.76rem; font-weight:800; text-transform:uppercase; letter-spacing:.06em; margin-bottom:.2rem; }\n      .credit-name { color:#172033; font-size:1.05rem; font-weight:750; }\n      @media (max-width:700px) { .project-credits-page { grid-template-columns:1fr; } }\n      .professor { display:flex; gap:1rem; align-items:center; background:white; padding:1rem; border-radius:18px; border:1px solid #e9e7ff; }\n      .avatar { width:68px; height:68px; border-radius:50%; display:flex; align-items:center; justify-content:center;\n        font-size:2rem; background:linear-gradient(135deg,#6f4bf2,#13c8c8); color:white; }\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\n\nCURRICULUM: dict[str, list[dict[str, Any]]] = {\n    "Entry": [\n        {\n            "id": "E1",\n            "title": "Bits, Qubits, and the Quantum Mindset",\n            "duration": 35,\n            "difficulty": 1,\n            "objectives": ["Compare a classical bit with a qubit", "Read ket notation", "Identify realistic quantum-computing use cases"],\n            "summary": "A bit is either 0 or 1. A qubit is described by a normalized complex state vector and can be measured to produce a classical result.",\n            "activity": "Classify five everyday problems as classical, quantum-inspired, or potentially quantum-relevant.",\n            "recording": "Recorded lesson: From bits to qubits",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/single-systems/introduction",\n        },\n        {\n            "id": "E2",\n            "title": "Superposition and Measurement",\n            "duration": 45,\n            "difficulty": 1,\n            "objectives": ["Explain amplitudes and probabilities", "Apply normalization", "Interpret repeated measurement counts"],\n            "summary": "Superposition is a linear combination of basis states. Measurement converts amplitudes into outcome probabilities through the Born rule.",\n            "activity": "Use the Quantum Coin mini-game to compare predicted and sampled outcomes.",\n            "recording": "Recorded lesson: Why quantum measurement is probabilistic",\n            "resource": "https://qiskit.qotlabs.org/learning/courses/basics-of-quantum-information/single-systems/quantum-information",\n        },\n        {\n            "id": "E3",\n            "title": "Quantum Gates and Circuit Diagrams",\n            "duration": 50,\n            "difficulty": 1,\n            "objectives": ["Recognize X, H, Z, and rotation gates", "Read a circuit left to right", "Predict basic single-qubit outcomes"],\n            "summary": "Quantum gates are reversible linear operations. The X gate flips basis states, while the H gate creates and removes equal superpositions.",\n            "activity": "Build a single-qubit circuit and compare your prediction with Qiskit simulation.",\n            "recording": "Recorded lesson: Gate-by-gate circuit reading",\n            "resource": "https://qiskit.qotlabs.org/docs/guides/circuit-library",\n        },\n        {\n            "id": "E4",\n            "title": "Your First Qiskit Program",\n            "duration": 55,\n            "difficulty": 1,\n            "objectives": ["Create a QuantumCircuit", "Simulate a statevector", "Sample measurement counts"],\n            "summary": "Qiskit represents quantum programs as circuits. A local statevector simulator is ideal for learning before using real quantum hardware.",\n            "activity": "Create a Bell state and explain why 01 and 10 should not appear in the ideal simulator.",\n            "recording": "Recorded lab: Build and run a Bell circuit",\n            "resource": "https://qiskit.qotlabs.org/docs/guides/quick-start",\n        },\n    ],\n    "Intermediate": [\n        {\n            "id": "I1",\n            "title": "Multiple Qubits and Tensor Products",\n            "duration": 60,\n            "difficulty": 2,\n            "objectives": ["Determine multi-qubit state dimensions", "Interpret basis ordering", "Construct product states"],\n            "summary": "The state space grows as 2^n for n qubits. Tensor products combine individual systems into a joint mathematical description.",\n            "activity": "Calculate the eight basis states for a three-qubit register.",\n            "recording": "Recorded lesson: Scaling from one qubit to many",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/multiple-systems/introduction",\n        },\n        {\n            "id": "I2",\n            "title": "Entanglement and Bell States",\n            "duration": 65,\n            "difficulty": 2,\n            "objectives": ["Distinguish correlation from entanglement", "Prepare a Bell state", "Interpret joint measurements"],\n            "summary": "Entangled states cannot be written as a product of independent subsystem states. Bell states are the standard two-qubit examples.",\n            "activity": "Run the Bell-state lab for several shot counts and compare statistical variation.",\n            "recording": "Recorded lesson: Entanglement without faster-than-light messaging",\n            "resource": "https://qiskit.qotlabs.org/learning/courses/basics-of-quantum-information/entanglement-in-action/introduction",\n        },\n        {\n            "id": "I3",\n            "title": "Teleportation and Superdense Coding",\n            "duration": 75,\n            "difficulty": 2,\n            "objectives": ["Describe the role of shared entanglement", "Track classical communication", "Separate state transfer from matter transfer"],\n            "summary": "Quantum teleportation transfers an unknown quantum state using shared entanglement plus two classical bits; it does not transmit matter or violate causality.",\n            "activity": "Draw the information flow and label quantum versus classical channels.",\n            "recording": "Recorded lesson: Quantum communication protocols",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/quantum-teleportation",\n        },\n        {\n            "id": "I4",\n            "title": "Quantum Algorithms: Deutsch–Jozsa and Grover",\n            "duration": 85,\n            "difficulty": 2,\n            "objectives": ["Explain oracle-based algorithms", "Describe amplitude amplification", "Compare query complexity"],\n            "summary": "Early quantum algorithms reveal how interference can suppress wrong answers and amplify useful ones. Grover search gives a quadratic query improvement for unstructured search.",\n            "activity": "Trace one Grover iteration for a two-qubit search space.",\n            "recording": "Recorded lesson: Interference as an algorithmic resource",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms",\n        },\n    ],\n    "Advanced": [\n        {\n            "id": "A1",\n            "title": "Density Matrices, Noise, and Channels",\n            "duration": 90,\n            "difficulty": 3,\n            "objectives": ["Represent mixed states", "Calculate purity conceptually", "Describe common noise channels"],\n            "summary": "Density matrices describe pure and mixed states. Quantum channels model physically allowed transformations, including noise and decoherence.",\n            "activity": "Compare ideal and noisy state descriptions and identify which information is lost.",\n            "recording": "Recorded seminar: From statevectors to open systems",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/general-formulation-of-quantum-information",\n        },\n        {\n            "id": "A2",\n            "title": "Variational Algorithms and VQE",\n            "duration": 100,\n            "difficulty": 3,\n            "objectives": ["Describe hybrid quantum-classical loops", "Identify ansatz and optimizer roles", "Interpret expectation-value objectives"],\n            "summary": "Variational algorithms use a parameterized quantum circuit and a classical optimizer. VQE estimates low-energy states of a Hamiltonian.",\n            "activity": "Sketch a VQE workflow and identify possible sources of optimization failure.",\n            "recording": "Recorded lab: Anatomy of a VQE experiment",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/quantum-chem-with-vqe",\n        },\n        {\n            "id": "A3",\n            "title": "QAOA and Combinatorial Optimization",\n            "duration": 100,\n            "difficulty": 3,\n            "objectives": ["Map a cost function to a Hamiltonian", "Explain alternating operators", "Evaluate approximation quality"],\n            "summary": "QAOA alternates problem and mixer operators. It is a framework for experimenting with approximate solutions to combinatorial problems.",\n            "activity": "Encode a small Max-Cut instance and discuss measurement-to-solution post-processing.",\n            "recording": "Recorded workshop: Hybrid optimization with QAOA",\n            "resource": "https://qiskit.qotlabs.org/learning/courses/variational-algorithm-design",\n        },\n        {\n            "id": "A4",\n            "title": "Quantum Error Correction and Fault Tolerance",\n            "duration": 110,\n            "difficulty": 3,\n            "objectives": ["Explain logical versus physical qubits", "Describe syndrome measurement", "Identify the purpose of fault tolerance"],\n            "summary": "Quantum error correction protects logical information by encoding it across many physical qubits and extracting error syndromes without directly reading the logical state.",\n            "activity": "Work through a three-qubit repetition-code example and identify its limitations.",\n            "recording": "Recorded seminar: Protecting fragile quantum information",\n            "resource": "https://quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction",\n        },\n    ],\n}\n\nQUIZZES: dict[str, list[dict[str, Any]]] = {\n    "Entry": [\n        {"q": "Which statement best describes a qubit before measurement?", "options": ["It is always secretly 0 or 1", "It can be represented by normalized complex amplitudes", "It stores two classical bits", "It violates probability rules"], "answer": 1, "why": "A pure qubit state is represented by two normalized complex amplitudes."},\n        {"q": "What does an ideal Hadamard gate do to |0>?", "options": ["Creates an equal superposition of |0> and |1>", "Always produces |1>", "Measures the qubit", "Copies the qubit"], "answer": 0, "why": "H|0> = (|0> + |1>)/sqrt(2)."},\n        {"q": "Why do repeated measurements of the same prepared superposition vary?", "options": ["The computer is broken", "Quantum outcomes are sampled from a probability distribution", "The circuit changes itself", "Qubits are classical random-number generators"], "answer": 1, "why": "Measurement samples outcomes according to the state\'s probability amplitudes."},\n        {"q": "Which gate acts like a classical NOT on basis states?", "options": ["Z", "H", "X", "S"], "answer": 2, "why": "The Pauli-X gate maps |0> to |1> and |1> to |0>."},\n    ],\n    "Intermediate": [\n        {"q": "How many computational basis states does a three-qubit register have?", "options": ["3", "6", "8", "9"], "answer": 2, "why": "An n-qubit system has 2^n basis states, so 2^3 = 8."},\n        {"q": "An entangled two-qubit state can always be written as…", "options": ["A product of two single-qubit states", "A classical probability table only", "A joint state that may not factor into subsystem states", "Two copied unknown qubits"], "answer": 2, "why": "Non-factorability is a defining feature of pure-state entanglement."},\n        {"q": "Quantum teleportation requires shared entanglement and…", "options": ["No communication", "Two classical bits", "Faster-than-light signaling", "A copy of the unknown state"], "answer": 1, "why": "The protocol requires two classical bits, preserving causality."},\n        {"q": "Grover\'s algorithm is primarily associated with…", "options": ["Unstructured search", "Sorting in constant time", "Copying quantum states", "Perfect error correction"], "answer": 0, "why": "Grover search provides quadratic query improvement for unstructured search."},\n    ],\n    "Advanced": [\n        {"q": "A density matrix is especially useful for representing…", "options": ["Only classical bits", "Mixed states and subsystems", "Only deterministic algorithms", "Copied unknown states"], "answer": 1, "why": "Density matrices represent both pure and mixed quantum states."},\n        {"q": "In VQE, the classical optimizer updates…", "options": ["Hardware temperature", "Parameters of an ansatz circuit", "The number of physical laws", "Measurement postulates"], "answer": 1, "why": "VQE iteratively adjusts parameterized circuit values to reduce an energy objective."},\n        {"q": "The purpose of a quantum error syndrome is to…", "options": ["Read the logical state directly", "Identify information about errors without revealing the logical data", "Eliminate all physical noise", "Clone the logical qubit"], "answer": 1, "why": "Syndromes reveal error information while preserving encoded logical information."},\n        {"q": "A fault-tolerant protocol is designed so that…", "options": ["One fault does not uncontrollably spread into many logical errors", "Every gate is noiseless", "No redundancy is needed", "Classical control is forbidden"], "answer": 0, "why": "Fault tolerance constrains error propagation and supports reliable logical operations."},\n    ],\n}\n\nKNOWLEDGE_BASE = [\n    {"title": "Qubit", "level": "Entry", "text": "A qubit is a two-level quantum information unit. A pure state can be written alpha|0> + beta|1>, where alpha and beta are complex amplitudes and their squared magnitudes sum to one."},\n    {"title": "Measurement", "level": "Entry", "text": "Computational-basis measurement returns 0 or 1. The probabilities are the squared magnitudes of the relevant amplitudes. Measurement also changes the post-measurement state."},\n    {"title": "Hadamard gate", "level": "Entry", "text": "The Hadamard gate creates equal superpositions from computational-basis states and can also recombine amplitudes, making interference visible."},\n    {"title": "No-cloning theorem", "level": "Intermediate", "text": "An arbitrary unknown quantum state cannot be perfectly copied by a universal physical operation. This is consistent with linear quantum evolution."},\n    {"title": "Entanglement", "level": "Intermediate", "text": "Entanglement describes joint quantum states whose correlations cannot be explained by assigning independent pure states to each subsystem. It does not enable faster-than-light communication."},\n    {"title": "Quantum teleportation", "level": "Intermediate", "text": "Teleportation transfers an unknown quantum state using one shared entangled pair, a joint measurement, and two classical bits. The sender\'s original state is not retained."},\n    {"title": "Grover search", "level": "Intermediate", "text": "Grover\'s algorithm alternates an oracle phase operation and diffusion-like amplitude amplification. It uses on the order of the square root of N oracle queries for an unstructured search space of size N."},\n    {"title": "Density matrix", "level": "Advanced", "text": "A density matrix is positive semidefinite with trace one. It represents pure states, probabilistic mixtures, and reduced states of larger entangled systems."},\n    {"title": "VQE", "level": "Advanced", "text": "The variational quantum eigensolver evaluates expectation values from a parameterized ansatz and uses a classical optimizer to search for a low-energy parameter setting."},\n    {"title": "Quantum error correction", "level": "Advanced", "text": "Quantum error correction encodes logical information into a larger Hilbert space. Syndrome measurements identify error information while avoiding direct measurement of the logical state."},\n    {"title": "QAOA", "level": "Advanced", "text": "QAOA alternates parameterized cost and mixer unitaries, then samples candidate solutions. Performance depends on encoding, depth, parameter optimization, noise, and post-processing."},\n]\n\nTEACHING_ARMS = ["Visual analogy", "Worked example", "Guided practice", "Short knowledge check", "Tutor session"]\n\n\ndef init_state() -> None:\n    defaults = {\n        "completed_lessons": [],\n        "quiz_history": [],\n        "xp": 120,\n        "streak": 4,\n        "bookings": [],\n        "chat_history": [\n            {"role": "assistant", "content": "Welcome! I am Professor Qubit. Ask me about qubits, gates, entanglement, algorithms, noise, or error correction."}\n        ],\n        "bandit_values": {arm: 0.5 for arm in TEACHING_ARMS},\n        "bandit_counts": {arm: 0 for arm in TEACHING_ARMS},\n        "current_strategy": "Visual analogy",\n        "last_lab_success": None,\n        "weekly_activity": pd.DataFrame(\n            {\n                "week": [f"W{i}" for i in range(1, 9)],\n                "minutes": [70, 95, 80, 135, 120, 155, 145, 180],\n                "quiz_accuracy": [55, 62, 65, 71, 74, 78, 80, 84],\n            }\n        ),\n    }\n    for key, value in defaults.items():\n        if key not in st.session_state:\n            st.session_state[key] = value\n\n\n@st.cache_resource\ndef train_ai_models() -> tuple[RandomForestRegressor, Pipeline, pd.DataFrame, float]:\n    rng = np.random.default_rng(42)\n    n = 2400\n    df = pd.DataFrame(\n        {\n            "weekly_hours": rng.uniform(0.5, 12.0, n),\n            "lessons_completed": rng.integers(0, 13, n),\n            "quiz_average": rng.uniform(25, 100, n),\n            "streak_days": rng.integers(0, 31, n),\n            "positive_feedback": rng.uniform(0.25, 1.0, n),\n            "math_comfort": rng.integers(1, 6, n),\n            "live_sessions": rng.integers(0, 7, n),\n            "difficulty_level": rng.integers(1, 4, n),\n        }\n    )\n    raw_progress = (\n        1.8 * df["weekly_hours"]\n        + 2.6 * df["lessons_completed"]\n        + 0.22 * df["quiz_average"]\n        + 0.38 * df["streak_days"]\n        + 8.0 * df["positive_feedback"]\n        + 2.0 * df["math_comfort"]\n        + 2.8 * df["live_sessions"]\n        - 3.0 * df["difficulty_level"]\n        + rng.normal(0, 5.5, n)\n    )\n    df["four_week_progress"] = np.clip(raw_progress, 3, 100)\n\n    risk_score = (\n        1.2\n        - 0.22 * df["weekly_hours"]\n        - 0.045 * df["quiz_average"]\n        - 0.08 * df["streak_days"]\n        - 0.7 * df["positive_feedback"]\n        - 0.25 * df["live_sessions"]\n        + 0.35 * df["difficulty_level"]\n        + rng.normal(0, 0.6, n)\n    )\n    df["support_risk"] = (risk_score > -2.55).astype(int)\n\n    features = [\n        "weekly_hours",\n        "lessons_completed",\n        "quiz_average",\n        "streak_days",\n        "positive_feedback",\n        "math_comfort",\n        "live_sessions",\n        "difficulty_level",\n    ]\n    progress_model = RandomForestRegressor(\n        n_estimators=220,\n        max_depth=10,\n        min_samples_leaf=3,\n        random_state=42,\n        n_jobs=-1,\n    )\n    progress_model.fit(df[features], df["four_week_progress"])\n\n    risk_model = Pipeline(\n        [\n            ("scale", StandardScaler()),\n            (\n                "dnn",\n                MLPClassifier(\n                    hidden_layer_sizes=(64, 32, 16),\n                    activation="relu",\n                    alpha=0.001,\n                    max_iter=500,\n                    random_state=42,\n                    early_stopping=True,\n                ),\n            ),\n        ]\n    )\n    risk_model.fit(df[features], df["support_risk"])\n    training_accuracy = float(risk_model.score(df[features], df["support_risk"]))\n    importance = pd.DataFrame(\n        {"feature": features, "importance": progress_model.feature_importances_}\n    ).sort_values("importance", ascending=False)\n    return progress_model, risk_model, importance, training_accuracy\n\n\n@st.cache_resource\ndef build_retriever() -> tuple[TfidfVectorizer, np.ndarray]:\n    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))\n    matrix = vectorizer.fit_transform([item["title"] + " " + item["text"] for item in KNOWLEDGE_BASE])\n    return vectorizer, matrix\n\n\ndef get_quiz_average() -> float:\n    if not st.session_state.quiz_history:\n        return 65.0\n    return float(np.mean([x["score_pct"] for x in st.session_state.quiz_history]))\n\n\ndef positive_feedback_rate() -> float:\n    total = sum(st.session_state.bandit_counts.values())\n    if total == 0:\n        return 0.65\n    weighted = sum(\n        st.session_state.bandit_values[a] * st.session_state.bandit_counts[a]\n        for a in TEACHING_ARMS\n    )\n    return float(weighted / total)\n\n\ndef learner_features(level: str, weekly_hours: float, math_comfort: int) -> pd.DataFrame:\n    return pd.DataFrame(\n        [\n            {\n                "weekly_hours": weekly_hours,\n                "lessons_completed": len(st.session_state.completed_lessons),\n                "quiz_average": get_quiz_average(),\n                "streak_days": st.session_state.streak,\n                "positive_feedback": positive_feedback_rate(),\n                "math_comfort": math_comfort,\n                "live_sessions": len(st.session_state.bookings),\n                "difficulty_level": LEVEL_TO_NUM[level],\n            }\n        ]\n    )\n\n\ndef choose_teaching_strategy(epsilon: float = 0.16) -> str:\n    if random.random() < epsilon:\n        return random.choice(TEACHING_ARMS)\n    values = st.session_state.bandit_values\n    max_value = max(values.values())\n    best = [arm for arm, value in values.items() if math.isclose(value, max_value, rel_tol=1e-9)]\n    return random.choice(best)\n\n\ndef update_bandit(arm: str, reward: int) -> None:\n    st.session_state.bandit_counts[arm] += 1\n    n = st.session_state.bandit_counts[arm]\n    old = st.session_state.bandit_values[arm]\n    st.session_state.bandit_values[arm] = old + (reward - old) / n\n    st.session_state.current_strategy = choose_teaching_strategy()\n\n\ndef professor_answer(question: str, learner_level: str, strategy: str) -> str:\n    vectorizer, matrix = build_retriever()\n    query_vec = vectorizer.transform([question])\n    scores = cosine_similarity(query_vec, matrix).ravel()\n    top_idx = scores.argsort()[::-1][:3]\n    top = [KNOWLEDGE_BASE[i] for i in top_idx]\n    confidence = float(scores[top_idx[0]]) if len(top_idx) else 0.0\n    core = top[0]["text"]\n    related = ", ".join(item["title"] for item in top[1:])\n\n    strategy_text = {\n        "Visual analogy": "Analogy: Think of amplitudes as arrows whose directions matter; when arrows combine, they can reinforce or cancel.",\n        "Worked example": "Worked example: Start with |0>, apply one gate at a time, and write the state after each operation before predicting measurement outcomes.",\n        "Guided practice": "Guided practice: First state the number of qubits, then list basis states, then identify gates, and only then calculate or simulate.",\n        "Short knowledge check": "Knowledge check: What result would change if the final measurement were removed? Explain your answer in one sentence.",\n        "Tutor session": "Tutor-session suggestion: Bring one circuit and one specific point of confusion to a 25-minute live session for targeted feedback.",\n    }[strategy]\n\n    level_note = {\n        "Entry": "I will keep the explanation conceptual and use minimal linear algebra.",\n        "Intermediate": "I will connect the concept to circuit behavior and basic state-vector reasoning.",\n        "Advanced": "I will connect the concept to formal representations, assumptions, and implementation tradeoffs.",\n    }[learner_level]\n\n    if confidence < 0.08:\n        caveat = "I found only a weak match in the built-in knowledge base, so treat this as a starting point and verify it with the linked course materials or a tutor."\n    else:\n        caveat = "This answer is grounded in the platform\'s curated concept notes."\n\n    return (\n        f"**Core idea — {top[0][\'title\']}**\\n\\n{core}\\n\\n"\n        f"**Adaptive teaching move**\\n\\n{strategy_text}\\n\\n"\n        f"**For your level**\\n\\n{level_note}\\n\\n"\n        f"**Related concepts:** {related}.\\n\\n"\n        f"*{caveat}*"\n    )\n\n\ndef find_recommended_lesson(level: str) -> dict[str, Any]:\n    for lesson in CURRICULUM[level]:\n        if lesson["id"] not in st.session_state.completed_lessons:\n            return lesson\n    for fallback in LEVELS:\n        for lesson in CURRICULUM[fallback]:\n            if lesson["id"] not in st.session_state.completed_lessons:\n                return lesson\n    return CURRICULUM["Advanced"][-1]\n\n\ndef make_circuit(experiment: str, theta: float) -> tuple[QuantumCircuit, str]:\n    if experiment == "Quantum coin (H gate)":\n        qc = QuantumCircuit(1)\n        qc.h(0)\n        expectation = "0 and 1 should be approximately balanced."\n    elif experiment == "Bell entanglement":\n        qc = QuantumCircuit(2)\n        qc.h(0)\n        qc.cx(0, 1)\n        expectation = "Only 00 and 11 should appear in the ideal simulator."\n    elif experiment == "Three-qubit GHZ":\n        qc = QuantumCircuit(3)\n        qc.h(0)\n        qc.cx(0, 1)\n        qc.cx(1, 2)\n        expectation = "Only 000 and 111 should appear in the ideal simulator."\n    else:\n        qc = QuantumCircuit(1)\n        qc.ry(theta, 0)\n        expectation = "The probability of 1 is sin²(theta/2)."\n    return qc, expectation\n\n\ndef sample_statevector(qc: QuantumCircuit, shots: int, seed: int = 7) -> tuple[dict[str, int], Statevector]:\n    state = Statevector.from_instruction(qc)\n    state.seed(seed)\n    counts = dict(state.sample_counts(shots=shots))\n    return counts, state\n\n\ndef score_quiz(level: str, selected: list[str]) -> tuple[int, list[dict[str, Any]]]:\n    questions = QUIZZES[level]\n    correct = 0\n    details = []\n    for item, answer_text in zip(questions, selected):\n        selected_idx = item["options"].index(answer_text)\n        is_correct = selected_idx == item["answer"]\n        correct += int(is_correct)\n        details.append({"question": item["q"], "correct": is_correct, "explanation": item["why"]})\n    return correct, details\n\n\ndef header(title: str, subtitle: str) -> None:\n    st.markdown(\n        f\'<div class="hero"><h1>{title}</h1><p>{subtitle}</p></div>\',\n        unsafe_allow_html=True,\n    )\n\n\ninit_state()\nprogress_model, risk_model, feature_importance, dnn_training_accuracy = train_ai_models()\n\nwith st.sidebar:\n    st.markdown("# ⚛️ QubitPath AI")\n    st.caption("Adaptive quantum-computing education prototype")\n    st.markdown(\n        f"""\n        <div class="project-credits-sidebar">\n          <div><span class="credit-label">Author</span><br><strong>{AUTHOR_NAME}</strong></div>\n          <div class="mentor-credit"><span class="credit-label">Mentor</span><br><strong>{MENTOR_NAME}</strong></div>\n        </div>\n        """,\n        unsafe_allow_html=True,\n    )\n    learner_name = st.text_input("Learner display name", value="Quantum Explorer")\n    learner_level = st.selectbox("Current pathway", LEVELS)\n    weekly_hours = st.slider("Planned study hours per week", 1.0, 12.0, 4.0, 0.5)\n    math_comfort = st.slider("Math comfort", 1, 5, 3, help="1 = developing; 5 = very comfortable")\n    learner_goal = st.selectbox(\n        "Primary goal",\n        ["Understand foundations", "Build Qiskit projects", "Prepare for research", "Explore quantum careers"],\n    )\n    st.markdown("---")\n    page = st.radio(\n        "Navigate",\n        ["Home", "Recorded Learning", "Live Tutoring", "Quantum Lab", "Quiz & Games", "AI Professor", "Analytics", "Responsible AI"],\n    )\n    st.markdown("---")\n    completion_pct = 100 * len(st.session_state.completed_lessons) / sum(len(v) for v in CURRICULUM.values())\n    st.progress(completion_pct / 100, text=f"Overall curriculum: {completion_pct:.0f}%")\n    st.caption(f"🔥 {st.session_state.streak}-day streak · ⭐ {st.session_state.xp} XP")\n\nfeatures = learner_features(learner_level, weekly_hours, math_comfort)\npredicted_progress = float(progress_model.predict(features)[0])\nrisk_probability = float(risk_model.predict_proba(features)[0, 1])\nrisk_label = "Needs support" if risk_probability >= 0.58 else "On track"\nrecommended = find_recommended_lesson(learner_level)\n\nif page == "Home":\n    header("Learn quantum computing with an adaptive guide", "Structured pathways, hands-on Qiskit labs, live tutoring, and transparent AI recommendations.")\n    st.markdown(f"### Welcome, {learner_name} 👋")\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric("Current pathway", learner_level)\n    c2.metric("4-week progress forecast", f"{predicted_progress:.0f}%")\n    c3.metric("Learning status", risk_label, f"{risk_probability:.0%} support probability")\n    c4.metric("Experience points", st.session_state.xp)\n\n    st.markdown("### Your next best action")\n    left, right = st.columns([1.55, 1])\n    with left:\n        st.markdown(\n            f"""\n            <div class="glass-card">\n              <span class="pill">{recommended[\'id\']}</span><span class="pill">{learner_level}</span>\n              <h3>{recommended[\'title\']}</h3>\n              <p>{recommended[\'summary\']}</p>\n              <p class="small-muted">Estimated time: {recommended[\'duration\']} minutes · Goal: {learner_goal}</p>\n            </div>\n            """,\n            unsafe_allow_html=True,\n        )\n    with right:\n        st.markdown(\n            f"""\n            <div class="glass-card">\n              <h3>🧠 Adaptive strategy</h3>\n              <p><b>{st.session_state.current_strategy}</b></p>\n              <p>The contextual bandit updates this recommendation when you rate Professor Qubit\'s help.</p>\n            </div>\n            """,\n            unsafe_allow_html=True,\n        )\n\n    st.markdown("### Explore the platform")\n    cols = st.columns(4)\n    cards = [\n        ("🎬", "Recorded learning", "Study leveled modules, lesson notes, activities, and official references."),\n        ("🧑\u200d🏫", "Live tutoring", "Book a prototype Zoom, Teams, or Google Meet session with a tutor."),\n        ("🧪", "Quantum lab", "Build and simulate Qiskit circuits with sampled quantum measurements."),\n        ("📊", "Analytics", "Review progress, skill coverage, quiz history, and AI forecasts."),\n    ]\n    for col, (icon, title, text) in zip(cols, cards):\n        with col:\n            st.markdown(f\'<div class="glass-card"><h2>{icon}</h2><h3>{title}</h3><p>{text}</p></div>\', unsafe_allow_html=True)\n\n    st.markdown("### How the AI components work")\n    st.info(\n        "Machine learning forecasts progress; a multilayer neural network estimates whether additional support may be helpful; "\n        "a transparent retrieval tutor answers from curated notes; and a bandit learns which teaching format receives positive feedback."\n    )\n    st.warning("All predictions are demonstrations trained on synthetic data. A human tutor should review any learner-support decision.")\n\nelif page == "Recorded Learning":\n    header("Recorded learning library", "Choose a pathway, open a lesson, complete its activity, and track your progress.")\n    tabs = st.tabs(LEVELS)\n    for tab, level in zip(tabs, LEVELS):\n        with tab:\n            st.markdown(f"### {level} pathway")\n            lessons = CURRICULUM[level]\n            selected_title = st.selectbox(\n                f"Select a {level.lower()} lesson",\n                [f"{x[\'id\']} — {x[\'title\']}" for x in lessons],\n                key=f"lesson_select_{level}",\n            )\n            lesson = lessons[[f"{x[\'id\']} — {x[\'title\']}" for x in lessons].index(selected_title)]\n            a, b = st.columns([1.6, 1])\n            with a:\n                st.subheader(lesson["title"])\n                st.caption(f"{lesson[\'duration\']} minutes · Difficulty {lesson[\'difficulty\']}/3")\n                st.markdown("**Learning objectives**")\n                for objective in lesson["objectives"]:\n                    st.markdown(f"- {objective}")\n                st.markdown("**Lesson notes**")\n                st.write(lesson["summary"])\n                st.markdown("**Hands-on activity**")\n                st.write(lesson["activity"])\n                st.markdown("**Recording panel**")\n                st.info(f"🎬 {lesson[\'recording\']} — prototype slot. Replace the external resource link with your own hosted recording when available.")\n                st.link_button("Open official learning resource", lesson["resource"], width="stretch")\n            with b:\n                completed = lesson["id"] in st.session_state.completed_lessons\n                st.markdown(\'<div class="glass-card">\', unsafe_allow_html=True)\n                st.markdown("#### Lesson checkpoint")\n                st.write("Explain the key concept in your own words, complete the activity, then mark the lesson complete.")\n                if completed:\n                    st.success("Completed ✓")\n                elif st.button("Mark lesson complete", key=f"complete_{lesson[\'id\']}", width="stretch"):\n                    st.session_state.completed_lessons.append(lesson["id"])\n                    st.session_state.xp += 40\n                    st.success("Lesson completed. +40 XP")\n                    st.rerun()\n                st.markdown("#### Recommended references")\n                st.markdown("- IBM Quantum Learning course catalog")\n                st.markdown("- Qiskit documentation and tutorials")\n                st.markdown("- *Quantum Computing for Everyone* — Chris Bernhardt")\n                st.markdown("- *Introduction to Classical and Quantum Computing* — Thomas Wong")\n                st.markdown(\'</div>\', unsafe_allow_html=True)\n\nelif page == "Live Tutoring":\n    header("Live tutoring studio", "Schedule guided practice with a tutor through Zoom, Microsoft Teams, or Google Meet.")\n    today = date.today()\n    schedule = pd.DataFrame(\n        [\n            {"Tutor": "Dr. Maya Chen", "Focus": "Foundations and Qiskit", "Date": today + timedelta(days=1), "Time": "5:00 PM", "Seats": 3},\n            {"Tutor": "Alex Rivera", "Focus": "Entanglement and algorithms", "Date": today + timedelta(days=2), "Time": "7:00 PM", "Seats": 2},\n            {"Tutor": "Dr. Samir Patel", "Focus": "VQE, QAOA, and research", "Date": today + timedelta(days=4), "Time": "6:30 PM", "Seats": 1},\n            {"Tutor": "Jordan Lee", "Focus": "Quantum math clinic", "Date": today + timedelta(days=5), "Time": "4:30 PM", "Seats": 4},\n        ]\n    )\n    st.dataframe(schedule, hide_index=True, width="stretch")\n\n    st.markdown("### Book a prototype session")\n    with st.form("booking_form"):\n        c1, c2 = st.columns(2)\n        tutor = c1.selectbox("Tutor", schedule["Tutor"].tolist())\n        platform = c2.selectbox("Meeting platform", ["Zoom", "Microsoft Teams", "Google Meet"])\n        session_date = c1.date_input("Preferred date", value=today + timedelta(days=2), min_value=today)\n        session_time = c2.time_input("Preferred time", value=datetime.strptime("18:00", "%H:%M").time())\n        topic = st.text_area("Topic or circuit you want help with", placeholder="Example: I understand H and CNOT separately, but I need help interpreting Bell-state measurement results.")\n        submitted = st.form_submit_button("Book demo session", width="stretch")\n    if submitted:\n        booking = {\n            "Tutor": tutor,\n            "Platform": platform,\n            "Date": str(session_date),\n            "Time": session_time.strftime("%I:%M %p"),\n            "Topic": topic or "General quantum-computing guidance",\n            "Status": "Requested",\n        }\n        st.session_state.bookings.append(booking)\n        st.session_state.xp += 15\n        st.success("Session request saved in this browser session. +15 XP")\n\n    if st.session_state.bookings:\n        st.markdown("### Your session requests")\n        st.dataframe(pd.DataFrame(st.session_state.bookings), hide_index=True, width="stretch")\n    st.markdown(\n        \'<div class="warning-box"><b>Production note:</b> Real meeting creation requires OAuth and the provider APIs. \'\n        \'Store credentials in a secret manager, collect the minimum data needed, and obtain consent before recording sessions.</div>\',\n        unsafe_allow_html=True,\n    )\n\nelif page == "Quantum Lab":\n    header("Qiskit quantum laboratory", "Create an ideal circuit, inspect its state, and compare your prediction with sampled measurement counts.")\n    left, right = st.columns([1, 1.25])\n    with left:\n        experiment = st.selectbox("Experiment", ["Quantum coin (H gate)", "Bell entanglement", "Three-qubit GHZ", "Custom Ry rotation"])\n        theta = st.slider("Rotation angle θ", 0.0, float(2 * np.pi), float(np.pi / 2), 0.05, disabled=experiment != "Custom Ry rotation")\n        shots = st.select_slider("Measurement shots", options=[100, 256, 512, 1024, 2048, 4096], value=1024)\n        prediction = st.selectbox(\n            "Predict the dominant ideal outcome pattern",\n            ["Mostly 0", "Mostly 1", "Balanced outcomes", "Only correlated all-zero/all-one outcomes"],\n        )\n        run_lab = st.button("Run quantum experiment", type="primary", width="stretch")\n        st.caption("The simulator is ideal and does not include hardware noise.")\n\n    qc, expectation = make_circuit(experiment, theta)\n    with right:\n        st.markdown("#### Circuit")\n        fig = qc.draw(output="mpl", fold=-1)\n        st.pyplot(fig, clear_figure=True)\n        plt.close(fig)\n\n    if run_lab:\n        counts, state = sample_statevector(qc, shots)\n        probability_df = pd.DataFrame({"State": list(counts.keys()), "Counts": list(counts.values())}).sort_values("State")\n        c1, c2 = st.columns([1.2, 1])\n        with c1:\n            bar = px.bar(probability_df, x="State", y="Counts", text="Counts", title="Sampled measurement counts")\n            bar.update_layout(yaxis_title="Counts", xaxis_title="Computational basis state")\n            st.plotly_chart(bar, width="stretch")\n        with c2:\n            st.markdown("#### Statevector amplitudes")\n            amplitude_rows = []\n            n_qubits = qc.num_qubits\n            for idx, amp in enumerate(state.data):\n                amplitude_rows.append(\n                    {\n                        "basis": format(idx, f"0{n_qubits}b"),\n                        "real": float(np.real(amp)),\n                        "imag": float(np.imag(amp)),\n                        "probability": float(abs(amp) ** 2),\n                    }\n                )\n            st.dataframe(pd.DataFrame(amplitude_rows), hide_index=True, width="stretch")\n            st.info(expectation)\n\n        expected_prediction = {\n            "Quantum coin (H gate)": "Balanced outcomes",\n            "Bell entanglement": "Only correlated all-zero/all-one outcomes",\n            "Three-qubit GHZ": "Only correlated all-zero/all-one outcomes",\n            "Custom Ry rotation": "Mostly 0" if theta < np.pi / 2 or theta > 3 * np.pi / 2 else ("Mostly 1" if np.pi / 2 < theta < 3 * np.pi / 2 else "Balanced outcomes"),\n        }[experiment]\n        success = prediction == expected_prediction\n        if success and st.session_state.last_lab_success != (experiment, prediction, round(theta, 2)):\n            st.session_state.xp += 25\n            st.session_state.last_lab_success = (experiment, prediction, round(theta, 2))\n            st.success("Your prediction matched the ideal pattern. +25 XP")\n        elif success:\n            st.success("Your prediction matched the ideal pattern.")\n        else:\n            st.warning(f"Compare your prediction with this expectation: {expectation}")\n\nelif page == "Quiz & Games":\n    header("Quizzes and mini-games", "Check understanding immediately and turn practice into an active learning loop.")\n    quiz_level = st.selectbox("Quiz level", LEVELS, index=LEVELS.index(learner_level))\n    with st.form(f"quiz_{quiz_level}"):\n        selected_answers = []\n        for idx, item in enumerate(QUIZZES[quiz_level], start=1):\n            selected_answers.append(st.radio(f"{idx}. {item[\'q\']}", item["options"], key=f"{quiz_level}_{idx}"))\n        quiz_submit = st.form_submit_button("Submit quiz", width="stretch")\n    if quiz_submit:\n        correct, details = score_quiz(quiz_level, selected_answers)\n        score_pct = 100 * correct / len(QUIZZES[quiz_level])\n        st.session_state.quiz_history.append(\n            {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "level": quiz_level, "score_pct": score_pct}\n        )\n        gained = int(10 * correct)\n        st.session_state.xp += gained\n        st.metric("Quiz score", f"{correct}/{len(QUIZZES[quiz_level])}", f"+{gained} XP")\n        for detail in details:\n            if detail["correct"]:\n                st.success(f"✓ {detail[\'question\']} — {detail[\'explanation\']}")\n            else:\n                st.error(f"Review: {detail[\'question\']} — {detail[\'explanation\']}")\n\n    st.markdown("### Gate-matching mini-game")\n    game_col1, game_col2 = st.columns(2)\n    challenge = game_col1.selectbox("Target transformation", ["Turn |0> into |1>", "Create equal probabilities from |0>", "Add a phase flip to |1>"])\n    chosen_gate = game_col2.selectbox("Choose a gate", ["X", "H", "Z"])\n    solution = {"Turn |0> into |1>": "X", "Create equal probabilities from |0>": "H", "Add a phase flip to |1>": "Z"}[challenge]\n    if st.button("Check gate", width="stretch"):\n        if chosen_gate == solution:\n            st.success("Correct. The selected gate matches the target transformation.")\n        else:\n            st.warning(f"Try again. The best match is the {solution} gate.")\n\nelif page == "AI Professor":\n    header("Professor Qubit", "A transparent AI tutor that retrieves curated concepts and adapts its teaching format from feedback.")\n    st.markdown(\n        \'<div class="professor"><div class="avatar">⚛️</div><div><h3>Professor Qubit</h3>\'\n        \'<p>Ask a quantum question. I will cite the closest built-in concept, tailor the depth to your pathway, and show the teaching strategy being tested.</p></div></div>\',\n        unsafe_allow_html=True,\n    )\n    st.caption(f"Current adaptive strategy: {st.session_state.current_strategy}")\n\n    for message in st.session_state.chat_history:\n        with st.chat_message(message["role"]):\n            st.markdown(message["content"])\n\n    prompt = st.chat_input("Ask Professor Qubit a question…")\n    if prompt:\n        st.session_state.chat_history.append({"role": "user", "content": prompt})\n        answer = professor_answer(prompt, learner_level, st.session_state.current_strategy)\n        st.session_state.chat_history.append({"role": "assistant", "content": answer})\n        st.rerun()\n\n    if len(st.session_state.chat_history) > 1:\n        st.markdown("#### Was the most recent teaching approach useful?")\n        f1, f2, f3 = st.columns([1, 1, 3])\n        if f1.button("👍 Helpful", width="stretch"):\n            update_bandit(st.session_state.current_strategy, 1)\n            st.success("Feedback recorded. The adaptive policy has been updated.")\n        if f2.button("👎 Not helpful", width="stretch"):\n            update_bandit(st.session_state.current_strategy, 0)\n            st.info("Feedback recorded. The platform will explore another teaching format.")\n        with f3:\n            st.caption("This is an epsilon-greedy contextual-bandit demonstration, not autonomous self-evolution.")\n\n    with st.expander("See the current bandit learning table"):\n        bandit_df = pd.DataFrame(\n            {\n                "Teaching strategy": TEACHING_ARMS,\n                "Estimated reward": [st.session_state.bandit_values[a] for a in TEACHING_ARMS],\n                "Feedback count": [st.session_state.bandit_counts[a] for a in TEACHING_ARMS],\n            }\n        )\n        st.dataframe(bandit_df, hide_index=True, width="stretch")\n\nelif page == "Analytics":\n    header("Learning analytics dashboard", "Review behavior, outcomes, forecasts, and model inputs without treating AI output as a final decision.")\n    m1, m2, m3, m4 = st.columns(4)\n    m1.metric("Lessons completed", len(st.session_state.completed_lessons), f"of {sum(len(v) for v in CURRICULUM.values())}")\n    m2.metric("Quiz average", f"{get_quiz_average():.0f}%")\n    m3.metric("4-week forecast", f"{predicted_progress:.0f}%")\n    m4.metric("Support status", risk_label, f"{risk_probability:.0%}")\n\n    c1, c2 = st.columns(2)\n    with c1:\n        activity_fig = px.line(\n            st.session_state.weekly_activity,\n            x="week",\n            y=["minutes", "quiz_accuracy"],\n            markers=True,\n            title="Weekly activity and quiz accuracy",\n        )\n        activity_fig.update_layout(legend_title_text="Metric")\n        st.plotly_chart(activity_fig, width="stretch")\n    with c2:\n        forecast_weeks = np.arange(0, 5)\n        current_completion = 100 * len(st.session_state.completed_lessons) / sum(len(v) for v in CURRICULUM.values())\n        projected = np.clip(current_completion + forecast_weeks * predicted_progress / 4.0, 0, 100)\n        forecast_df = pd.DataFrame({"Week": forecast_weeks, "Projected curriculum completion": projected})\n        forecast_fig = px.area(forecast_df, x="Week", y="Projected curriculum completion", title="Illustrative completion trajectory")\n        forecast_fig.update_yaxes(range=[0, 100], ticksuffix="%")\n        st.plotly_chart(forecast_fig, width="stretch")\n\n    c3, c4 = st.columns(2)\n    with c3:\n        completed_set = set(st.session_state.completed_lessons)\n        skill_scores = {\n            "Foundations": min(100, 25 + 15 * len(completed_set.intersection({"E1", "E2", "E3", "E4"}))),\n            "Circuits": min(100, 20 + 18 * len(completed_set.intersection({"E3", "E4", "I2"}))),\n            "Algorithms": min(100, 15 + 22 * len(completed_set.intersection({"I4", "A2", "A3"}))),\n            "Quantum math": min(100, 20 + 16 * len(completed_set.intersection({"I1", "A1"}))),\n            "Error correction": min(100, 10 + 55 * int("A4" in completed_set)),\n        }\n        radar = go.Figure()\n        radar.add_trace(go.Scatterpolar(r=list(skill_scores.values()), theta=list(skill_scores.keys()), fill="toself", name="Skill score"))\n        radar.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])), showlegend=False, title="Skill coverage profile")\n        st.plotly_chart(radar, width="stretch")\n    with c4:\n        if st.session_state.quiz_history:\n            quiz_df = pd.DataFrame(st.session_state.quiz_history)\n            quiz_fig = px.bar(quiz_df, x="date", y="score_pct", color="level", title="Quiz history", range_y=[0, 100])\n            st.plotly_chart(quiz_fig, width="stretch")\n        else:\n            st.info("Complete a quiz to populate the quiz-history chart.")\n\n    st.markdown("### Why the progress model made its forecast")\n    importance_fig = px.bar(feature_importance.sort_values("importance"), x="importance", y="feature", orientation="h", title="Random-forest global feature importance on synthetic training data")\n    st.plotly_chart(importance_fig, width="stretch")\n    st.caption(f"The support-risk neural network uses hidden layers (64, 32, 16). Synthetic-data training accuracy: {dnn_training_accuracy:.1%}. This number does not validate real-world performance.")\n\n    report = {\n        "learner": learner_name,\n        "pathway": learner_level,\n        "goal": learner_goal,\n        "generated_at": datetime.now().isoformat(timespec="seconds"),\n        "lessons_completed": st.session_state.completed_lessons,\n        "quiz_average": get_quiz_average(),\n        "xp": st.session_state.xp,\n        "four_week_progress_forecast": predicted_progress,\n        "support_risk_probability": risk_probability,\n        "support_status": risk_label,\n        "recommended_next_lesson": recommended["id"],\n    }\n    r1, r2 = st.columns(2)\n    r1.download_button("Download learner report (JSON)", json.dumps(report, indent=2), file_name="qubitpath_learner_report.json", mime="application/json", width="stretch")\n    csv_report = pd.DataFrame([report | {"lessons_completed": ",".join(report["lessons_completed"])}]).to_csv(index=False)\n    r2.download_button("Download learner report (CSV)", csv_report, file_name="qubitpath_learner_report.csv", mime="text/csv", width="stretch")\n\nelse:\n    header("Responsible AI and production roadmap", "Build learner trust through transparency, privacy, accessibility, evaluation, and human oversight.")\n    st.markdown("### Project credits")\n    st.markdown(\n        f"""\n        <div class="project-credits-page">\n          <div class="credit-card">\n            <div class="credit-role">Author</div>\n            <div class="credit-name">{AUTHOR_NAME}</div>\n          </div>\n          <div class="credit-card">\n            <div class="credit-role">Mentor</div>\n            <div class="credit-name">{MENTOR_NAME}</div>\n          </div>\n        </div>\n        """,\n        unsafe_allow_html=True,\n    )\n    st.markdown("### What this prototype does")\n    st.markdown(\n        """\n        - Uses **synthetic data** to demonstrate progress forecasting and learner-support risk prediction.\n        - Uses a **curated retrieval tutor** rather than pretending that a generic language model is always correct.\n        - Uses a small **epsilon-greedy bandit** to learn which teaching format receives positive feedback.\n        - Keeps all session data in Streamlit\'s temporary browser/server session state.\n        """\n    )\n    st.markdown("### What a production platform still needs")\n    roadmap = pd.DataFrame(\n        [\n            ["Identity and roles", "Student, tutor, parent/guardian, instructor, and administrator permissions", "High"],\n            ["Persistent database", "Encrypted learner profiles, progress events, content, and audit logs", "High"],\n            ["Live-session APIs", "OAuth integration for Zoom, Teams, or Google Meet", "High"],\n            ["Content management", "Instructor upload, captioning, transcripts, versioning, and moderation", "High"],\n            ["Model evaluation", "Real holdout data, subgroup analysis, calibration, drift monitoring, and tutor review", "High"],\n            ["Accessibility", "Keyboard navigation, captions, transcripts, readable contrast, and screen-reader testing", "High"],\n            ["Security", "Secret management, least privilege, rate limiting, backups, and incident response", "High"],\n            ["Optional LLM tutor", "Grounded generation, citation checks, refusal behavior, and age-appropriate controls", "Medium"],\n            ["Real quantum hardware", "IBM Quantum credentials, job limits, cost controls, and queue feedback", "Medium"],\n        ],\n        columns=["Capability", "Production requirement", "Priority"],\n    )\n    st.dataframe(roadmap, hide_index=True, width="stretch")\n\n    st.markdown("### Guardrails")\n    g1, g2, g3 = st.columns(3)\n    with g1:\n        st.markdown(\'<div class="glass-card"><h3>🔒 Privacy</h3><p>Minimize data collection, encrypt records, define retention limits, and obtain consent for recordings.</p></div>\', unsafe_allow_html=True)\n    with g2:\n        st.markdown(\'<div class="glass-card"><h3>🧑\u200d🏫 Human oversight</h3><p>Never use a risk score as an automatic grade or penalty. Let tutors review context and learner preferences.</p></div>\', unsafe_allow_html=True)\n    with g3:\n        st.markdown(\'<div class="glass-card"><h3>📐 Evaluation</h3><p>Measure learning gains, calibration, false alerts, accessibility, and performance across relevant learner groups.</p></div>\', unsafe_allow_html=True)\n\n    st.markdown("### Official learning references")\n    ref1, ref2 = st.columns(2)\n    ref1.link_button("IBM Quantum Learning catalog", IBM_COURSES_URL, width="stretch")\n    ref2.link_button("Qiskit documentation", QISKIT_DOCS_URL, width="stretch")\n'


## 5. Create deployment requirements and verify the app syntax

In [ ]:
REQUIREMENTS = """streamlit==1.59.2
qiskit[visualization]~=2.5.0
scikit-learn>=1.5,<2
plotly>=5.24,<7
pandas>=2.2,<3
numpy>=1.26,<3
matplotlib>=3.8,<4
joblib>=1.4,<2
"""
from pathlib import Path
Path("requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
compile(Path("app.py").read_text(encoding="utf-8"), "app.py", "exec")
print("Created requirements.txt")
print("Syntax check passed for app.py")



## 6. Launch the Streamlit app in Google Colab

This cell starts Streamlit on port 8501 and exposes it through a temporary Cloudflare tunnel. The public URL appears in the output after the tunnel starts.

- Keep this cell running while using the app.
- The temporary URL changes when the runtime restarts.
- The tunnel is for prototyping only, not production deployment.


In [ ]:

import os
import subprocess
import time
from pathlib import Path

# Stop older prototype processes if this launch cell is rerun.
os.system("pkill -f 'streamlit run app.py' >/dev/null 2>&1 || true")
os.system("pkill -f 'cloudflared tunnel' >/dev/null 2>&1 || true")

# Download Cloudflare's temporary tunnel client if needed.
if not Path("cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

streamlit_log = open("streamlit.log", "w")
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)
time.sleep(5)

print("Streamlit started. A temporary public URL will appear below.\n")
!./cloudflared tunnel --url http://127.0.0.1:8501 --no-autoupdate



## 7. Download the generated project files from Colab

Run this optional cell after `app.py` and `requirements.txt` have been created.


In [ ]:

from google.colab import files
import zipfile
from pathlib import Path

zip_name = "qubitpath_ai_streamlit_project.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("app.py")
    zf.write("requirements.txt")

print("Created:", zip_name)
files.download(zip_name)



## 8. Suggested production extensions

1. Replace synthetic learner data with consented, de-identified event data and a documented schema.
2. Add a secure database such as PostgreSQL and role-based access control.
3. Connect Zoom, Teams, or Google Meet through OAuth rather than storing credentials in code.
4. Add instructor content upload, captions, transcripts, content moderation, and version control.
5. Evaluate forecasting and support-risk models on real holdout data; monitor calibration, subgroup error, and drift.
6. Add grounded LLM generation only after implementing source citations, prompt-injection defenses, age-appropriate controls, and tutor escalation.
7. Connect IBM Quantum hardware through supported credentials, queue controls, quotas, and cost limits.
8. Deploy the Streamlit app from a GitHub repository or a managed cloud platform instead of a temporary notebook tunnel.
